# SBAS-интерферометрия пашни Татарстана: низкая когерентность, стек за последние 5 лет

Этот ноутбук адаптирован под **слабокогерентные сельскохозяйственные поля Татарстана** и опирается на рабочие паттерны из примеров PyGMTSAR (`goldenvalley`, `lakesarez_landslides_2017`, `imperial_valley_2015`). Основная идея — **максимально агрессивно стабилизировать шумные данные**, но при этом сохранить связный временной ряд за **последние 5 лет от даты запуска ноутбука**.

## Практические эвристики для пашни

- **Зима в целом исключается**: декабрь, январь и март обычно только разрушают сеть интерферограмм.
- **Февраль оставляем** как «мост» между годами.
- **Осень — основной сезон**: сентябрь, октябрь, ноябрь.
- Для слабой когерентности используются:
  - burst-обработка вместо полных сцен;
  - укороченные SBAS-пары внутри осени;
  - отдельные длинные «связующие» пары через февраль;
  - multilooking и PS-веса;
  - маскирование по корреляции перед SNAPHU;
  - отбор главной связной компоненты после unwrap.

> **Важно:** замените примерный `AOI` ниже на контур вашей пашни. Чем точнее полигон, тем стабильнее подбор burst-стека и тем меньше лишнего шума.


In [ ]:
import os
import subprocess
import sys

if 'google.colab' in sys.modules:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pygmtsar'])
    import importlib.resources as resources
    with resources.as_file(resources.files('pygmtsar.data') / 'google_colab.sh') as script:
        subprocess.check_call(['sh', os.fspath(script)])
    from google.colab import output
    output.enable_custom_widget_manager()


In [ ]:
import warnings

import dask
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from dask.distributed import Client
from shapely.geometry import Polygon

from pygmtsar import ASF, S1, Stack, Tiles, tqdm_dask

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['figure.dpi'] = 140
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', None)


In [ ]:
# Даты задаются динамически: 5 последних лет от даты запуска.
END_DATE = pd.Timestamp.utcnow().tz_localize(None).normalize()
START_DATE = END_DATE - pd.DateOffset(years=5)

# Примерный контур пашни в Татарстане.
# ОБЯЗАТЕЛЬНО замените координаты на свой фактический полигон.
AOI = gpd.GeoDataFrame(
    geometry=[Polygon([
        (50.2106, 55.0018),
        (50.2522, 55.0018),
        (50.2522, 54.9787),
        (50.2106, 54.9787),
        (50.2106, 55.0018),
    ])],
    crs='EPSG:4326',
)

AUTUMN_MONTHS = (9, 10, 11)
BRIDGE_MONTH = 2
ALLOWED_MONTHS = (BRIDGE_MONTH, *AUTUMN_MONTHS)

WORKDIR = 'raw_tatarstan_farmland_sbas'
DATADIR = 'data_tatarstan_farmland_sbas'
DEM = f'{DATADIR}/dem.nc'

PAIR_DAYS_AUTUMN = 48     # короткие пары для слабокогерентной пашни
PAIR_DAYS_BRIDGE = 420    # длинные связи через февраль между годами
PAIR_METERS = 250         # мягкое ограничение по B_perp
INTF_RESOLUTION = 90      # более грубое разрешение для устойчивости
INTF_WAVELENGTH = 180     # агрессивнее фильтрация/многократный look
CORR_THRESHOLD = 0.12     # стартовый порог; при разрывах можно ослабить до 0.08

print(f'Окно обработки: {START_DATE.date()} — {END_DATE.date()}')
AOI


In [ ]:
def prepare_search_results(search_gdf):
    gdf = search_gdf.copy()
    gdf['date'] = pd.to_datetime(gdf['startTime']).dt.normalize()
    gdf['month'] = gdf['date'].dt.month
    gdf['year'] = gdf['date'].dt.year
    gdf['orbit_code'] = gdf['flightDirection'].map({
        'ASCENDING': 'A',
        'DESCENDING': 'D',
    }).fillna(gdf['flightDirection'].astype(str).str[0])
    return gdf


def rank_tracks(search_gdf, allowed_months=ALLOWED_MONTHS):
    gdf = prepare_search_results(search_gdf)
    gdf = gdf[gdf['month'].isin(allowed_months)].copy()
    rows = []
    for (orbit_code, path_number), group in gdf.groupby(['orbit_code', 'pathNumber']):
        date_table = group[['date', 'month']].drop_duplicates()
        autumn_dates = int(date_table[date_table['month'].isin(AUTUMN_MONTHS)]['date'].nunique())
        feb_dates = int(date_table[date_table['month'] == BRIDGE_MONTH]['date'].nunique())
        total_dates = int(date_table['date'].nunique())
        total_bursts = int(group['fileID'].nunique())
        rows.append({
            'orbit_code': orbit_code,
            'pathNumber': int(path_number),
            'autumn_dates': autumn_dates,
            'february_dates': feb_dates,
            'total_dates': total_dates,
            'total_bursts': total_bursts,
            'score': autumn_dates * 100 + feb_dates * 25 + total_dates,
        })
    ranked = pd.DataFrame(rows).sort_values(
        ['score', 'autumn_dates', 'february_dates', 'total_bursts'],
        ascending=False,
    )
    return ranked.reset_index(drop=True)


def keep_consistent_dates(track_gdf):
    gdf = prepare_search_results(track_gdf)
    coverage = gdf.groupby('date')['fileID'].nunique().sort_index()
    target = int(coverage.mode().iloc[0])
    keep_dates = coverage[coverage >= max(1, target - 1)].index
    return gdf[gdf['date'].isin(keep_dates)].copy(), coverage


def choose_reference_date(track_gdf):
    gdf = prepare_search_results(track_gdf)
    dates = pd.Series(sorted(gdf['date'].drop_duplicates()))
    preferred = dates[dates.dt.month.isin(AUTUMN_MONTHS)]
    if preferred.empty:
        preferred = dates[dates.dt.month == BRIDGE_MONTH]
    if preferred.empty:
        preferred = dates
    return preferred.iloc[len(preferred) // 2].strftime('%Y-%m-%d')


def build_farmland_pairs(sbas, autumn_days=PAIR_DAYS_AUTUMN, bridge_days=PAIR_DAYS_BRIDGE, meters=PAIR_METERS):
    import pandas as pd

    all_pairs = sbas.sbas_pairs(days=bridge_days, meters=meters).copy()

    autumn = all_pairs[
        all_pairs['ref'].dt.month.isin(AUTUMN_MONTHS)
        & all_pairs['rep'].dt.month.isin(AUTUMN_MONTHS)
        & (all_pairs['duration'] <= autumn_days)
    ].copy()

    autumn_to_feb = all_pairs[
        all_pairs['ref'].dt.month.isin(AUTUMN_MONTHS)
        & (all_pairs['rep'].dt.month == BRIDGE_MONTH)
        & (all_pairs['rep'].dt.year == all_pairs['ref'].dt.year + 1)
    ].copy()
    autumn_to_feb['bridge_key'] = autumn_to_feb['rep'].dt.year
    autumn_to_feb = autumn_to_feb.sort_values(['bridge_key', 'duration', 'baseline']).groupby('bridge_key').head(2)

    feb_to_autumn = all_pairs[
        (all_pairs['ref'].dt.month == BRIDGE_MONTH)
        & all_pairs['rep'].dt.month.isin(AUTUMN_MONTHS)
        & (all_pairs['rep'].dt.year == all_pairs['ref'].dt.year)
    ].copy()
    feb_to_autumn['bridge_key'] = feb_to_autumn['ref'].dt.year
    feb_to_autumn = feb_to_autumn.sort_values(['bridge_key', 'duration', 'baseline']).groupby('bridge_key').head(2)

    feb_to_feb = all_pairs[
        (all_pairs['ref'].dt.month == BRIDGE_MONTH)
        & (all_pairs['rep'].dt.month == BRIDGE_MONTH)
        & (all_pairs['rep'].dt.year == all_pairs['ref'].dt.year + 1)
    ].copy()
    feb_to_feb['bridge_key'] = feb_to_feb['ref'].dt.year
    feb_to_feb = feb_to_feb.sort_values(['bridge_key', 'duration', 'baseline']).groupby('bridge_key').head(1)

    pairs = pd.concat([autumn, autumn_to_feb, feb_to_autumn, feb_to_feb], ignore_index=True)
    pairs = pairs.drop(columns=['bridge_key'], errors='ignore')
    pairs = pairs.drop_duplicates(subset=['ref', 'rep']).sort_values(['ref', 'rep']).reset_index(drop=True)

    filler = sbas.sbas_pairs_fill(pairs[['ref', 'rep']])
    if filler is not None:
        filler = filler.merge(
            all_pairs.drop(columns=['pair']).copy(),
            on=['ref', 'rep', 'duration'],
            how='left',
        )
        pairs = pd.concat([pairs, filler], ignore_index=True)
        pairs = pairs.drop_duplicates(subset=['ref', 'rep']).sort_values(['ref', 'rep']).reset_index(drop=True)

    assert len(pairs) > 0, 'Не удалось собрать сеть SBAS-пар. Проверьте AOI и доступные даты.'
    return pairs


## 1. Поиск burst-кандидатов за 5 лет

Ноутбук ищет burst'ы для **обоих направлений**, затем автоматически выбирает трек, где лучше соблюдается нужная нам сезонная схема:

1. много осенних дат;
2. есть февральские сцены для мостов между годами;
3. покрытие burst'ами достаточно стабильное по датам.


In [ ]:
search_all = ASF.search(
    AOI,
    startTime=str(START_DATE.date()),
    stopTime=str(END_DATE.date()),
    flightDirection=None,
)
search_all = prepare_search_results(search_all)
search_all = search_all[search_all['month'].isin(ALLOWED_MONTHS)].copy()

track_ranking = rank_tracks(search_all)
display(track_ranking.head(10))

best_track = track_ranking.iloc[0]
selected_track = search_all[
    (search_all['orbit_code'] == best_track['orbit_code'])
    & (search_all['pathNumber'] == best_track['pathNumber'])
].copy()
selected_track, coverage_per_date = keep_consistent_dates(selected_track)

REFERENCE = choose_reference_date(selected_track)
BURSTS = selected_track.sort_values(['date', 'fileID'])['fileID'].tolist()

print(f"Выбран трек: orbit={best_track['orbit_code']}, path={best_track['pathNumber']}")
print(f'Опорная дата: {REFERENCE}')
print(f'Burst-файлов к скачиванию: {len(BURSTS)}')
display(coverage_per_date.to_frame('bursts_per_date').tail(20))
selected_track[['date', 'month', 'pathNumber', 'orbit_code', 'fileID']].head()


In [ ]:
ax = selected_track.dissolve(by='date').boundary.plot(figsize=(8, 8), color='steelblue', linewidth=0.7)
AOI.boundary.plot(ax=ax, color='red', linewidth=2)
ax.set_title('Выбранные burst-контуры и AOI')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.show()


## 2. Скачивание данных

Ниже используются Earthdata / ASF credentials. При необходимости замените `None` на свои логин и пароль.


In [ ]:
asf_username = None
asf_password = None

asf = ASF(asf_username, asf_password)
print(asf.download(DATADIR, BURSTS))

scenes = S1.scan_slc(DATADIR)
S1.download_orbits(DATADIR, scenes)
Tiles().download_dem(AOI, filename=DEM)


## 3. Локальный Dask и инициализация стека


In [ ]:
if 'client' in globals():
    client.close()
client = Client()
client


In [ ]:
scenes = S1.scan_slc(DATADIR)
sbas = Stack(WORKDIR, drop_if_exists=True).set_scenes(scenes).set_reference(REFERENCE)
display(sbas.to_dataframe().head())
sbas.plot_scenes(AOI=AOI)


In [ ]:
# Для burst-стека обрезаем стек по AOI, чтобы убрать лишнее и уменьшить шум.
sbas.compute_reframe(AOI)
sbas.load_dem(DEM, AOI)
sbas.compute_align()
sbas.compute_geocode(1)

sbas.plot_topo(quantile=[0.01, 0.99])


## 4. Сезонно-устойчивая SBAS-сеть для пашни

Сеть строится так:

- короткие пары внутри осени (`<= 48` дней);
- 1–2 связи «осень → февраль следующего года»;
- 1–2 связи «февраль → осень того же года»;
- 1 связь «февраль → февраль следующего года».

Это дает компромисс между **устойчивостью к декорреляции** и **связностью ряда между годами**.


In [ ]:
pairs = build_farmland_pairs(sbas)
display(pairs[['ref', 'rep', 'duration', 'baseline']].head(30))
print(f'Всего SBAS-пар: {len(pairs)}')
sbas.plot_baseline(pairs)


## 5. PS-веса и multilook-интерферограммы

Для слабокогерентной пашни здесь специально усилена стабилизация:

- считаем **PS function** по стеку амплитуд;
- используем ее как вес в `compute_interferogram_multilook`;
- работаем на **грубее выходной сетке** (`90 м`) и с **увеличенным фильтрующим масштабом** (`180 м`).

Если сцены особенно шумные, сначала увеличьте `INTF_RESOLUTION` до `120`, а уже потом ослабляйте `CORR_THRESHOLD`.


In [ ]:
sbas.compute_ps()
psf = sbas.psfunction()
sbas.plot_psfunction(psf, quantile=[0.05, 0.95])


In [ ]:
sbas.compute_interferogram_multilook(
    pairs,
    'intf_farm_mlook',
    weight=psf,
    resolution=INTF_RESOLUTION,
    wavelength=INTF_WAVELENGTH,
    coarsen=(2, 8),
)

ds_sbas = sbas.open_stack('intf_farm_mlook')
intf_sbas = ds_sbas.phase
corr_sbas = ds_sbas.correlation

sbas.plot_interferograms(intf_sbas[:8], caption='Фаза SBAS, [rad]')
sbas.plot_correlations(corr_sbas[:8], caption='Корреляция SBAS')


## 6. Unwrap только по достаточно связным пикселям

Для пашни лучше сначала отбросить совсем развалившиеся области и уже потом запускать SNAPHU. После unwrap сохраняем только **главную связную компоненту**.


In [ ]:
unwrap_input = intf_sbas.where(corr_sbas >= CORR_THRESHOLD)
unwrap_weight = corr_sbas.where(corr_sbas >= CORR_THRESHOLD)

unwrap_sbas = sbas.unwrap_snaphu(
    unwrap_input,
    unwrap_weight,
    conncomp=True,
)
unwrap_sbas = sbas.sync_cube(unwrap_sbas, 'unwrap_farm_sbas')
unwrap_sbas = sbas.conncomp_main(unwrap_sbas, 1)

sbas.plot_phases((unwrap_sbas.phase - unwrap_sbas.phase.mean(['y', 'x']))[:8], caption='Unwrap после отбора главной компоненты')


## 7. Удаление трендов и оценка смещений/скоростей

Для длинного 5-летнего ряда на сельхозугодьях полезно убрать остаточные крупномасштабные тренды, связанные с топографией, геометрией обзора и атмосферой.


In [ ]:
decimator = sbas.decimator(resolution=30, grid=(1, 1))
topo = decimator(sbas.get_topo())
inc = decimator(sbas.incidence_angle())
yy, xx = xr.broadcast(topo.y, topo.x)

trend_sbas = sbas.regression(
    unwrap_sbas.phase,
    [
        topo,
        topo * yy,
        topo * xx,
        topo ** 2,
        inc,
        yy,
        xx,
        yy * xx,
    ],
    corr_sbas,
)
trend_sbas = sbas.sync_cube(trend_sbas, 'trend_farm_sbas')

sbas.plot_phases(trend_sbas[:8], caption='Оцененный тренд', quantile=[0.01, 0.99])
sbas.plot_phases((unwrap_sbas.phase - trend_sbas)[:8], caption='Фаза после detrend', vmin=-np.pi, vmax=np.pi)


In [ ]:
disp_sbas = sbas.los_displacement_mm(sbas.lstsq(unwrap_sbas.phase - trend_sbas, corr_sbas))
disp_sbas = sbas.sync_cube(disp_sbas, 'disp_farm_sbas')

velocity_sbas = sbas.velocity(disp_sbas)
velocity_sbas = sbas.sync_cube(velocity_sbas, 'velocity_farm_sbas')

sbas.plot_displacements(disp_sbas[:8], caption='Накопленное LOS-смещение, [mm]', quantile=[0.01, 0.99], symmetrical=True)


In [ ]:
velocity_ll = sbas.as_geo(sbas.ra2ll(velocity_sbas)).rio.clip(AOI.geometry)

zmin, zmax = np.nanquantile(velocity_ll, [0.01, 0.99])
span = max(abs(zmin), abs(zmax))
fig, ax = plt.subplots(figsize=(9, 6))
velocity_ll.plot.imshow(ax=ax, cmap='turbo', vmin=-span, vmax=span)
AOI.boundary.plot(ax=ax, color='black', linewidth=1)
ax.set_title('Скорость LOS, мм/год')
plt.show()

velocity_ll.to_netcdf('tatarstan_farmland_velocity.nc')
print('Сохранено: tatarstan_farmland_velocity.nc')


## 8. Что крутить в первую очередь, если сеть рвется

1. Убедиться, что AOI — это **реальный контур поля**, а не слишком большой прямоугольник.
2. Повысить `INTF_RESOLUTION` с `90` до `120` м.
3. Увеличить `INTF_WAVELENGTH` с `180` до `240` м.
4. Ослабить `CORR_THRESHOLD` с `0.12` до `0.10` или `0.08`.
5. Если осенних сцен мало, добавить **август** в `AUTUMN_MONTHS`, но не возвращать декабрь/январь/март, пока не доказано обратное.
6. Если связность между годами слабая, увеличить `PAIR_DAYS_BRIDGE` до `450` и оставить только 1 лучшую bridge-пару на год.

## Что крутить в первую очередь, если результат слишком сглажен

1. Уменьшить `INTF_RESOLUTION` до `60` м.
2. Уменьшить `INTF_WAVELENGTH` до `120` м.
3. Оставить осеннюю сеть как есть, но поднять `CORR_THRESHOLD` до `0.15`.
